In [ ]:
import math
from pathlib import Path

import torch
import torchmetrics
from lightning import Trainer, seed_everything
from lightning.pytorch.callbacks import BatchSizeFinder, ModelCheckpoint, RichModelSummary
from ovito.data import DataCollection
from ovito.io import export_file, import_file
from src import LightningDataset, Module
from src.constants import DEFAULT_SEED
from src.graph import PeriodicKNN
from src.transforms import RandomPerturbation
from src.typing import PathLike
from torch import Tensor
from torch_geometric.loader import NeighborLoader
from tqdm.auto import tqdm

In [ ]:
torch.serialization.add_safe_globals([RandomPerturbation])

In [ ]:
torch.set_float32_matmul_precision("medium")
torch.backends.cudnn.benchmark = True
seed_everything(DEFAULT_SEED)

In [ ]:
EPOCHS: int = 30
NUM_NEIGHBORS: int = 20
COMPILE: bool = True
CKPT_NAME: PathLike | None = Path(".") / "lightning_logs" / "version_0" / "checkpoints" / "epoch=22-step=782.ckpt"
TO_PREDICT: list[PathLike] = [
    # Path.home() / "THESE" / "TEST" / "Si_mixture_polycrystal" / "final.cfg",
]

In [ ]:
callbacks= [
    ModelCheckpoint(monitor="val/loss", mode="min", every_n_epochs=1),
    RichModelSummary(max_depth=2),
    BatchSizeFinder()
]

trainer = Trainer(
    max_epochs=EPOCHS,
    precision="bf16-mixed" if torch.cuda.is_available() else 32,
    callbacks=callbacks,
    enable_progress_bar=True,
    enable_model_summary=False,
)

datamodule = LightningDataset(
    dataset_name="custom",
    lengths=(0.7, 0.2, 0.1),
    transforms=RandomPerturbation(std_range=(0.0, 0.05)),
    num_workers=8,
    batch_size=512,
    k=NUM_NEIGHBORS,
    use_imbalance_sampler=True,
)

num_classes = datamodule.num_classes
metrics = torchmetrics.MetricCollection({
    "f1": torchmetrics.F1Score(task="multiclass", num_classes=num_classes),
    "auroc": torchmetrics.AUROC(task="multiclass", num_classes=num_classes),
    "acc": torchmetrics.Accuracy(task="multiclass", num_classes=num_classes),
})

should_train = CKPT_NAME is None or not Path(CKPT_NAME).exists()
if should_train:
    with trainer.init_module():
        model = Module(
            model_name="PaiNN",
            num_classes=num_classes,
            compile=COMPILE,
            metrics=metrics,
            warmup=400,
            lr=0.004678965862088063,
            max_iters=EPOCHS * len(datamodule.train_dataloader()),
            model_kwargs={
                "num_radial": 8,
                "hidden_channels": 32,
                "num_layers": 2,
                "dropout": 0.5,
                "scale_factor": 1.0 / math.sqrt(NUM_NEIGHBORS),
            },
        )
else:
    print(f"Loading checkpoint weights from: {CKPT_NAME}")
    with trainer.init_module(empty_init=True):
        model = Module.load_from_checkpoint(CKPT_NAME)

In [ ]:
if should_train:
    trainer.fit(model=model, datamodule=datamodule)
    trainer.validate(model=model, datamodule=datamodule)
    trainer.test(model=model, datamodule=datamodule)

In [ ]:
@torch.inference_mode()
def inference(model: Module, data: DataCollection) -> Tensor:
    model.eval()
    device = next(model.parameters()).device
    num_layers: int = model.model.num_layers  # type: ignore

    knn = PeriodicKNN(k=NUM_NEIGHBORS)
    graph = knn.convert(data)

    loader = NeighborLoader(
        graph,
        num_neighbors=[-1] * num_layers,
        batch_size=min(2**16, graph.num_nodes),  # type: ignore
        shuffle=False,
        num_workers=8,
        persistent_workers=True,
        pin_memory=True,
        prefetch_factor=2,
    )

    graph_preds = []
    for batch in tqdm(loader, unit="batch", total=len(loader)):
        batch = batch.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            out = model.predict_step(batch)
        graph_preds.append(out.to("cpu", non_blocking=True))

    torch.cuda.synchronize()
    predictions = torch.cat(graph_preds, dim=0)

    return predictions


def dump_outputs(
    preds: Tensor,
    data: DataCollection,
    fpath: PathLike,
) -> None:
    path = Path(fpath)
    out_path = path.with_name(f"{path.stem}_predicted.extxyz")

    pred_array = preds.detach().cpu().numpy()
    data.particles_.create_property("Prediction", data=pred_array)

    export_file(
        data,
        str(out_path),
        "xyz",
        columns=[
            "Particle Identifier",
            "Particle Type",
            "Position.X",
            "Position.Y",
            "Position.Z",
            "Prediction",
        ],
    )
    print(f"Saved predictions to {out_path}")

In [ ]:
if TO_PREDICT:
    for path in tqdm(TO_PREDICT):
        data = import_file(path).compute()
        preds = inference(model, data)
        dump_outputs(preds, data, path)
else:
    print("No files specified for prediction.")